In [ ]:
%matplotlib widget

import numpy as np
import matplotlib.pyplot as plt

from ipywidgets import (
    FloatSlider, IntSlider,
    HTML, HTMLMath, VBox, HBox, Layout
)
from IPython.display import display

# ============================================================
# TWO-DIMENSIONAL WAVE EQUATION
#
# u_xx + u_yy = (1/c^2) u_tt
#
# Fixed square membrane:
#
# 0 < x < L
# 0 < y < L
#
# U_nm(x,y) =
# sin(n*pi*x/L) sin(m*pi*y/L)
#
# omega_nm =
# (c*pi/L) sqrt(n^2+m^2)
# ============================================================

plt.ioff()

display(HTML("""
<style>

.container { width:98% !important; max-width:none !important; }

.output_area,
.output_subarea,
.jp-Cell-outputWrapper,
.jp-OutputArea,
.jp-OutputArea-child,
.jp-OutputArea-output,
.widget-output,
.jupyter-widgets-output-area,
.widget-box {
    max-width:none !important;
    height:auto !important;
    max-height:none !important;
    overflow:visible !important;
}

.output_scroll {
    height:auto !important;
    max-height:none !important;
    overflow:visible !important;
    box-shadow:none !important;
}

.jupyter-matplotlib,
.jupyter-matplotlib-figure {
    overflow:visible !important;
    resize:none !important;
}

.wave-title {
    font-family:Arial;
    font-size:20px;
    font-weight:bold;
    color:#6f3fa0;
}

.wave-label {
    font-family:Arial;
    font-size:14px;
    font-weight:bold;
}

.wave-value {
    font-family:Arial;
    font-size:14px;
    font-weight:bold;
    color:#0b3d91;
}

</style>
"""))

# ============================================================
# DOCUMENTATION
# ============================================================

documentation = HTML("""
<div style="
    width:1180px;
    font-family:Arial;
    font-size:15px;
    line-height:1.48;
    margin-bottom:10px;
">

<div class="wave-title" style="margin-bottom:8px;">
Normal Modes of a Two-Dimensional Vibrating Membrane
</div>

<div style="margin-bottom:5px;">
For a square membrane with fixed boundaries, separation of variables
leads to spatial eigenfunctions that are products of sine functions.
</div>

<div style="margin-bottom:5px;">
Each pair of integers (n,m) defines one normal mode with angular
frequency ωₙₘ=(cπ/L)√(n²+m²).
</div>

<div>
<b>This notebook:</b> displays the spatial mode shape and its harmonic
time evolution for selected values of n and m.
</div>

</div>
""")

mode_math = HTMLMath(
    value=(
        r'\('
        r'U_{nm}(x,y)='
        r'\sin\bigg(\dfrac{n\pi x}{L}\bigg)'
        r'\sin\bigg(\dfrac{m\pi y}{L}\bigg)'
        r'\)'
    )
)

frequency_math = HTMLMath(
    value=(
        r'\('
        r'\omega_{nm}='
        r'\dfrac{c\pi}{L}'
        r'\sqrt{n^2+m^2}'
        r'\)'
    )
)

# ============================================================
# GRID
# ============================================================

L = 1.0
NX = 220

x = np.linspace(0.0, L, NX)
y = np.linspace(0.0, L, NX)

X, Y = np.meshgrid(x, y)

# ============================================================
# CONTROLS
# ============================================================

slider_style = {'description_width': '0px'}

n_slider = IntSlider(
    min=1, max=8, step=1, value=1,
    readout=False, continuous_update=True,
    style=slider_style, layout=Layout(width='220px')
)

m_slider = IntSlider(
    min=1, max=8, step=1, value=2,
    readout=False, continuous_update=True,
    style=slider_style, layout=Layout(width='220px')
)

c_slider = FloatSlider(
    min=0.5, max=3.0, step=0.1, value=1.0,
    readout=False, continuous_update=True,
    style=slider_style, layout=Layout(width='220px')
)

time_slider = FloatSlider(
    min=0.0, max=4.0, step=0.01, value=0.0,
    readout=False, continuous_update=True,
    style=slider_style, layout=Layout(width='220px')
)

n_value = HTML('<div class="wave-value">1</div>')
m_value = HTML('<div class="wave-value">2</div>')
c_value = HTML('<div class="wave-value">1.0</div>')
time_value = HTML('<div class="wave-value">0.00</div>')

def row(label, slider, value):
    return HBox(
        [
            HTML(
                f'<div class="wave-label">{label}</div>',
                layout=Layout(width='110px', min_width='110px')
            ),
            slider,
            value
        ],
        layout=Layout(width='420px', height='38px', align_items='center')
    )

controls = VBox(
    [
        HTML('<div class="wave-title" style="margin-bottom:7px;">Mode Parameters</div>'),
        row('Mode n:', n_slider, n_value),
        row('Mode m:', m_slider, m_value),
        row('Wave speed c:', c_slider, c_value),
        row('Time t:', time_slider, time_value)
    ],
    layout=Layout(
        width='455px',
        padding='10px 14px',
        border='1px solid #d2c2df'
    )
)

current_math = HTMLMath()

current_panel = VBox(
    [
        HTML('<div class="wave-title" style="margin-bottom:7px;">Current Mode</div>'),
        mode_math,
        frequency_math,
        current_math
    ],
    layout=Layout(
        width='660px',
        padding='10px 14px',
        border='1px solid #d2c2df'
    )
)

top_row = HBox(
    [controls, current_panel],
    layout=Layout(width='1140px', gap='15px', align_items='stretch')
)

# ============================================================
# INITIAL MODE
# ============================================================

def spatial_mode(n, m):
    return (
        np.sin(n*np.pi*X/L)
        *
        np.sin(m*np.pi*Y/L)
    )

n0 = n_slider.value
m0 = m_slider.value
c0 = c_slider.value
t0 = time_slider.value

U0 = spatial_mode(n0, m0)

omega0 = (
    c0*np.pi/L
    *
    np.sqrt(n0**2 + m0**2)
)

instantaneous0 = (
    U0
    *
    np.cos(omega0*t0)
)

# ============================================================
# FIGURE 1 — MEMBRANE
# ============================================================

fig_mode, ax_mode = plt.subplots(figsize=(6.0, 5.0))

fig_mode.canvas.header_visible = False
fig_mode.canvas.footer_visible = False
fig_mode.canvas.toolbar_visible = False
fig_mode.canvas.layout = Layout(width='600px', height='500px')

ax_mode.set_title(
    'Instantaneous Membrane Displacement',
    fontsize=14,
    fontweight='bold',
    color='#6f3fa0'
)

ax_mode.set_xlabel('x')
ax_mode.set_ylabel('y')

mode_image = ax_mode.imshow(
    instantaneous0,
    extent=[0.0, L, 0.0, L],
    origin='lower',
    aspect='equal',
    vmin=-1.0,
    vmax=1.0,
    interpolation='bilinear'
)

fig_mode.colorbar(
    mode_image,
    ax=ax_mode,
    fraction=0.046,
    pad=0.04
)

fig_mode.subplots_adjust(
    left=0.12, right=0.90,
    top=0.90, bottom=0.12
)

# ============================================================
# FIGURE 2 — TIME FACTOR
# ============================================================

fig_time, ax_time = plt.subplots(figsize=(5.2, 5.0))

fig_time.canvas.header_visible = False
fig_time.canvas.footer_visible = False
fig_time.canvas.toolbar_visible = False
fig_time.canvas.layout = Layout(width='520px', height='500px')

ax_time.set_title(
    'Temporal Factor',
    fontsize=14,
    fontweight='bold',
    color='#0b3d91'
)

ax_time.set_xlabel('Time t')
ax_time.set_ylabel('cos(ωₙₘt)')
ax_time.set_xlim(0.0, 4.0)
ax_time.set_ylim(-1.15, 1.15)
ax_time.grid(True, linestyle=':', alpha=0.40)

time_axis = np.linspace(0.0, 4.0, 1500)

time_curve, = ax_time.plot(
    time_axis,
    np.cos(omega0*time_axis),
    linewidth=2.0
)

time_marker = ax_time.axvline(
    t0,
    linestyle='--',
    linewidth=1.2
)

time_point, = ax_time.plot(
    [t0],
    [np.cos(omega0*t0)],
    linestyle='None',
    marker='o',
    markersize=7
)

fig_time.subplots_adjust(
    left=0.15, right=0.97,
    top=0.90, bottom=0.13
)

figures_row = HBox(
    [fig_mode.canvas, fig_time.canvas],
    layout=Layout(width='1135px', gap='10px', align_items='flex-start')
)

# ============================================================
# UPDATE
# ============================================================

def update_notebook(change=None):
    n = n_slider.value
    m = m_slider.value
    c = c_slider.value
    t = time_slider.value

    omega = (
        c*np.pi/L
        *
        np.sqrt(n**2 + m**2)
    )

    U = spatial_mode(n, m)

    instantaneous = (
        U
        *
        np.cos(omega*t)
    )

    mode_image.set_data(instantaneous)

    time_curve.set_ydata(
        np.cos(omega*time_axis)
    )

    time_marker.set_xdata([t, t])

    time_point.set_data(
        [t],
        [np.cos(omega*t)]
    )

    n_value.value = f'<div class="wave-value">{n}</div>'
    m_value.value = f'<div class="wave-value">{m}</div>'
    c_value.value = f'<div class="wave-value">{c:.1f}</div>'
    time_value.value = f'<div class="wave-value">{t:.2f}</div>'

    current_math.value = (
        r'\('
        r'\omega_{nm}='
        + f'{omega:.6f}'
        + r',\;'
        r'T_{nm}='
        + f'{2*np.pi/omega:.6f}'
        + r'\)'
    )

    fig_mode.canvas.draw_idle()
    fig_time.canvas.draw_idle()

n_slider.observe(update_notebook, names='value')
m_slider.observe(update_notebook, names='value')
c_slider.observe(update_notebook, names='value')
time_slider.observe(update_notebook, names='value')

update_notebook()

display(
    VBox(
        [
            documentation,
            top_row,
            figures_row
        ],
        layout=Layout(width='1180px', gap='10px')
    )
)